# 08 · Statistical API: five AOIs, two months

**Objective:** Produce a masked feature table with 10-day intervals, real projected geometry and valid-sample-weighted monthly composite summaries.

**Route:** Live API / CDSE OAuth client · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** interval and monthly-composite CSV tables plus plots. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/cdse-sentinel-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Define illustrative AOIs

These five small rectangles are teaching sites, not validated peatland polygons. Coordinates start in WGS84, then **the actual geometry is transformed** to EPSG:32635 before requesting 10 m resolution. This zone is an illustrative metric CRS for these nearby Finnish sites; choose a suitable local CRS for other regions.


In [ ]:
from examples.sentinel import access_token, statistics, statistic_rows, projected_geometry
from examples.climate import weighted_interval_mean

aois = {
    "finland_site_1": [24.55, 60.15, 24.57, 60.17],
    "finland_site_2": [24.80, 60.20, 24.82, 60.22],
    "finland_site_3": [25.05, 60.30, 25.07, 60.32],
    "finland_site_4": [24.30, 60.45, 24.32, 60.47],
    "finland_site_5": [25.30, 60.10, 25.32, 60.12],
}
periods = {"2024-07": ("2024-07-01T00:00:00Z", "2024-08-01T00:00:00Z"), "2024-08": ("2024-08-01T00:00:00Z", "2024-09-01T00:00:00Z")}
print(projected_geometry(aois["finland_site_1"]))


## 3. Retrieve ten requests

The evalscript outputs red, near infrared and NDVI plus a dataMask. The shortened final interval is retained explicitly, rather than silently dropping the last day. Review processing quotas before running ten requests.


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
token = access_token()
rows, requests_used = [], []
for site, bbox in aois.items():
    for period, (start, end) in periods.items():
        response, payload = statistics(token, bbox, start, end)
        rows.extend(statistic_rows(response, site, period))
        requests_used.append({"site": site, "period": period, "payload": payload})
table = pd.DataFrame(rows)
if table.empty:
    raise ValueError("No statistics returned; check request intervals and account access.")
table.head(10)


## 4. Weight interval means by valid samples

`sampleCount − noDataCount` is the valid count. A monthly mean of interval means without weights gives sparse and dense observations equal influence. The weighted output below is a **sample-weighted mosaic composite**, not a uniform-time monthly mean or an average across every acquisition.


In [ ]:
summary = []
for (site, period), group in table.groupby(["site", "period"]):
    row = {"site": site, "period": period}
    for band in ("red", "nir", "ndvi"):
        row[f"{band}_sample_weighted_mean"] = weighted_interval_mean(group[f"{band}_mean"], group[f"{band}_valid_count"])
        row[f"{band}_valid_count"] = int(group[f"{band}_valid_count"].sum())
    summary.append(row)
summary = pd.DataFrame(summary)
summary


In [ ]:
table["from"] = pd.to_datetime(table["from"], utc=True)
fig, ax = plt.subplots(figsize=(10, 4))
for site, group in table.groupby("site"):
    ax.plot(group["from"], group["ndvi_mean"], marker="o", label=site)
ax.set_ylabel("Masked interval composite NDVI")
ax.legend()
plt.show()
(ROOT / "outputs").mkdir(exist_ok=True)
table.to_csv(ROOT / "outputs" / "sentinel-2-statistical-intervals.csv", index=False)
summary.to_csv(ROOT / "outputs" / "sentinel-2-statistical-composites.csv", index=False)
record_provenance("sentinel-2-statistics.json", requests=requests_used, summary_semantics="valid-sample-weighted composite of 10-day mosaics", geometry_crs="EPSG:32635", requested_resolution_m=10, interval_policy="SHORTEN")


## Interpret and verify

Each interval is a mosaic using mostRecent ordering; these statistics are not a sample of every scene. SCL classes 4/5/6 are retained. Water has meaningful spectral NDVI but should be separated for vegetation-specific research. Retain counts and missing intervals in model features.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [CDSE Statistical API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Statistical.html)
- [CDSE API request reference](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/ApiReference.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
